# Happiness in the music we listen to, and how we respond to it
### By Quinn Epstein
---
## Setup for graphing

In [60]:
import plotly.graph_objects as go;
import opendatasets as od;
import pandas as pd;
from plotly.matplotlylib.mpltools import DASH_MAP

od.download('https://www.kaggle.com/datasets/ektanegi/spotifydata-19212020');
od.download('https://www.kaggle.com/datasets/gauthamvijayaraj/spotify-tracks-dataset-updated-every-week');
od.download('https://www.kaggle.com/datasets/khushikyad001/world-happiness-report');

spotifyLegacy = pd.read_csv('spotifydata-19212020/data.csv');
spotifyModern = pd.read_csv('spotify-tracks-dataset-updated-every-week/spotify_tracks.csv');

# Normalize headers
spotifyModern.columns = spotifyModern.columns.str.lower().str.strip();

# fixing inconsistent column names from data
for feature in ['valence', 'danceability']:
    if feature not in spotifyModern.columns:
        match = next((c for c in spotifyModern.columns if feature[:4] in c), None);
        if match: spotifyModern.rename(columns={match: feature}, inplace=True);

if 'year' not in spotifyModern.columns:
    dateCol = next((c for c in spotifyModern.columns if 'date' in c), None);
    spotifyModern['year'] = pd.to_datetime(spotifyModern[dateCol], errors='coerce').dt.year if dateCol else pd.NA;

# saving needed columns 
cols = ['year', 'valence', 'danceability'];
spotifyStats = pd.concat([spotifyLegacy[cols], spotifyModern[cols]], ignore_index=True);

spotifyStats = spotifyStats[(spotifyStats['year'] >= 1921) & (spotifyStats['year'] <= 2024)];
spotifyStats.dropna(subset=['year', 'valence', 'danceability'], inplace=True);

# score normalizing 
happinessScores = pd.read_csv('world-happiness-report/world_happiness_report.csv');
happinessScores.rename(columns={'Happiness_Score': 'Happiness Score'}, inplace=True);
happinessScores['NormalizedScores'] = happinessScores['Happiness Score'] / 10;

# group by year 
yearlyMusic = spotifyStats.groupby('year')[['valence', 'danceability']].mean().reset_index();
yearlyHappiness = happinessScores.groupby('Year')['NormalizedScores'].mean().reset_index();

Skipping, found downloaded files in ".\spotifydata-19212020" (use force=True to force download)
Skipping, found downloaded files in ".\spotify-tracks-dataset-updated-every-week" (use force=True to force download)
Skipping, found downloaded files in ".\world-happiness-report" (use force=True to force download)


---
## First im going to make two graphs showing happiness overtime 

In [61]:
happinessGraph = go.Figure();

happinessGraph.add_trace(go.Scatter(
    x = yearlyMusic['year'],
    y = yearlyMusic['valence'],
    mode = 'lines+markers',
    name = 'Music Happiness (Valence)',
    line = dict(color = '#785EF0', width = 3)));

happinessGraph.update_layout(
    title = 'Happiness of Music (1921 - 2024)',
    xaxis_title = 'Year',
    yaxis_title = 'Music Happiness (0.0 - 1.0)',
    template = 'ggplot2',
    hovermode = 'x unified',
    xaxis = dict(range = [1921, 2027])
);

happinessGraph.show();

## Plus World Happiness Data

In [62]:
happinessGraph.add_trace(go.Scatter(
    x = yearlyHappiness['Year'],
    y = yearlyHappiness['NormalizedScores'],
    mode = 'lines+markers',
    name = 'Global Happiness Score',
    line = dict(color = '#FE6100', width = 4, dash = 'dash')));

happinessGraph.update_layout(
    title = 'Music Happiness vs Global Happiness',
    yaxis_title = 'Happiness Score (0.0 - 1.0)',
    # Zooming in on the 2000s where we have the most correlation data
    xaxis = dict(range = [2000, 2027])
);

happinessGraph.show();

## finally we add how dancable the music is, for the large scope 

In [63]:
happinessGraph.add_trace(go.Scatter(
    x = yearlyMusic['year'],
    y = yearlyMusic['danceability'],
    mode = 'lines+markers',
    name = 'Danceability',
    line = dict(color = '#FFB000', width = 3, dash = 'dot')));

happinessGraph.update_layout(
    title = 'Relationship Between Music Vibe and Danceability',
    legend = dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    xaxis = dict(range = [1921, 2027])
);

happinessGraph.show();